# Notas — Aula 9: Herança e composição

Marco: nasce a primeira **família de robôs**. `RoboComSensor(Robo)` — herança,
`super().__init__`, sobrescrita de método — enxerga mais longe que um `Robo`
comum. Depois, o mesmo problema (sensor de alcance variável) resolvido **sem**
nenhuma subclasse: `Sensor` vira um objeto que `Robo` **tem**, e `sensor_frente()`
delega pra ele.

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**) — cada
> seção depende da anterior.

In [109]:
LADO_GRADE = 10

## Herança — `class Sub(Base)` e `super().__init__`

Herança em Python usa a sintaxe `class Sub(Base):`. Dentro do `__init__` da
subclasse, `super().__init__(...)` chama o `__init__` da classe-mãe — sem isso, os
atributos que o pai cria nunca existem na subclasse. No robô, usamos isso para
criar variações de `Robo` sem reescrever `nome`/`x`/`y` do zero em cada uma.

In [110]:
class Veiculo:
    def __init__(self, marca, modelo):
        self.marca = marca
        self.modelo = modelo


class Carro(Veiculo):
    def __init__(self, marca, modelo, portas):
        super().__init__(marca, modelo)
        self.portas = portas


c = Carro("Fiat", "Uno", 4)
print(c.marca, c.modelo, c.portas)

Fiat Uno 4


In [111]:
class Robo:
    def __init__(self, nome, x=0, y=0, obstaculos=None):
        self.nome = nome
        self.x = x
        self.y = y
        self.obstaculos = obstaculos if obstaculos is not None else {}


class RoboComSensor(Robo):
    def __init__(self, nome, alcance_sensor=2, **kwargs):
        super().__init__(nome, **kwargs)
        self.alcance_sensor = alcance_sensor


rs = RoboComSensor("R2D2")
print(rs.nome, rs.alcance_sensor)

R2D2 2


### Sua vez

`RoboNumerado` esquece de repassar `**kwargs` para `super().__init__` — por isso
`x`/`y` sempre voltam para o valor padrão, mesmo quando alguém passa um valor
diferente. Complete a chamada para repassar `**kwargs` também, não só `nome`.

*Dica: `super().__init__(nome, **kwargs)`, igual em `RoboComSensor` acima.*

In [112]:
class RoboNumerado(Robo):
    def __init__(self, nome, numero_serie, **kwargs):
        super().__init__(nome, kwargs)   # TODO: repasse **kwargs também, não só nome
        self.numero_serie = numero_serie


rn = RoboNumerado("Bender", numero_serie=42, x=5, y=3)
print(rn.nome, rn.numero_serie, rn.x, rn.y)

Bender 42 {'x': 5, 'y': 3} 0


## Sobrescrita: substituir vs. estender

Sobrescrever um método pode **substituir** por completo a versão do pai (a nova
versão nunca chama a antiga) ou **estender** — chamar `super().metodo()` e fazer
mais em cima. `RoboComSensor.sensor_frente()` substitui; `RoboVeloz.avancar()`
estende, chamando `super().avancar()` duas vezes.

In [113]:
class Veiculo:
    def __init__(self, marca, modelo):
        self.marca = marca
        self.modelo = modelo

    def info(self):
        return f"{self.marca} {self.modelo}"


class Moto(Veiculo):
    def __init__(self, marca, modelo, cilindradas):
        super().__init__(marca, modelo)
        self.cilindradas = cilindradas

    def info(self):
        return f"{super().info()} ({self.cilindradas}cc)"


m = Moto("Honda", "CG", 150)
print(m.info())

Honda CG (150cc)


In [114]:
class Robo:
    def __init__(self, nome, x=0):
        self.nome = nome
        self.x = x

    def avancar(self):
        self.x += 1
        return True


class RoboVeloz(Robo):
    def avancar(self):
        moveu1 = super().avancar()
        moveu2 = super().avancar()
        return moveu1 or moveu2


flash = RoboVeloz("Flash")
flash.avancar()
print(flash.x)

2


### Sua vez

Complete `RoboBarulhento.avancar()`: depois de chamar `super().avancar()`, se
`moveu` for `True`, imprima `f"{self.nome} avançou!"`.

*Dica: um `if moveu:` com um `print` dentro, antes do `return moveu`.*

In [115]:
class RoboBarulhento(Robo):
    def avancar(self):
        moveu = super().avancar()
        # TODO: se moveu for True, imprima f"{self.nome} avançou!"
        if moveu : print(f"{self.nome} avançou!")
        pass
        return moveu


rb = RoboBarulhento("Barulhento")
rb.avancar()
print(rb.x)

Barulhento avançou!
1


## O erro clássico: esquecer `super().__init__()`

Nada avisa na hora de **definir** a subclasse — o erro só aparece quando alguém
tenta **usar** um atributo que devia ter sido criado no `__init__` do pai, e ele
nunca rodou.

In [116]:
class Veiculo:
    def __init__(self, marca, modelo):
        self.marca = marca
        self.modelo = modelo

    def info(self):
        return f"{self.marca} {self.modelo}"


class MotoQuebrada(Veiculo):
    def __init__(self, marca, modelo, cilindradas):
        self.cilindradas = cilindradas    # esqueceu super().__init__(marca, modelo)


try:
    mq = MotoQuebrada("Honda", "CG", 150)
    print(mq.info())
except AttributeError as erro:
    print(f"AttributeError: {erro}")

AttributeError: 'MotoQuebrada' object has no attribute 'marca'


In [117]:
class Robo:
    def __init__(self, nome, x=0):
        self.nome = nome #Por ser um atributo de INSTANCIA, sem o super().__init__, suas subclasses não "possuirão" este atributo.
        self.x = x


class RoboComSensorQuebrado(Robo):
    def __init__(self, nome, alcance_sensor=2):
        self.alcance_sensor = alcance_sensor   # esqueceu super().__init__


try:
    r = RoboComSensorQuebrado("Bug")
    print(r.nome)
except AttributeError as erro:
    print(f"AttributeError: {erro}")

print(vars(RoboComSensorQuebrado("Bug2")))

AttributeError: 'RoboComSensorQuebrado' object has no attribute 'nome'
{'alcance_sensor': 2}


### Sua vez

`RoboIncompleto` tem o mesmo bug: guarda `cor` antes de chamar `super().__init__`.
Conserte, chamando `super().__init__(nome)` **antes** de `self.cor = cor`.

*Dica: troque a ordem das duas linhas dentro do `__init__`.*

In [118]:
class RoboIncompleto(Robo):
    def __init__(self, nome, cor):
        super().__init__(nome)
        self.cor = cor    # TODO: chame super().__init__(nome) ANTES desta linha


try:
    ri = RoboIncompleto("C3PO", "dourado")
    print(ri.nome, ri.cor)
except AttributeError as erro:
    print(f"AttributeError: {erro}")

C3PO dourado


## `isinstance()` — checando a relação é-um

Herança é uma relação **é-um**, de mão única: todo `RoboComSensor` é um `Robo`,
mas nem todo `Robo` é um `RoboComSensor`. `isinstance(objeto, Classe)` confirma
essa relação em tempo de execução.

In [119]:
class Robo:
    def __init__(self, nome):
        self.nome = nome


class RoboComSensor(Robo):
    def __init__(self, nome, alcance_sensor=2):
        super().__init__(nome)
        self.alcance_sensor = alcance_sensor


robos = [Robo("Wall-E"), RoboComSensor("R2D2"), RoboComSensor("C3PO")]
for r in robos:
    print(r.nome, isinstance(r, Robo), isinstance(r, RoboComSensor))

Wall-E True False
R2D2 True True
C3PO True True


### Sua vez

Complete `todos_sao_sensor(lista)`: devolva `True` se **todos** os itens da lista
forem `RoboComSensor` (não basta `Robo`), `False` caso contrário.

*Dica: `all(isinstance(r, RoboComSensor) for r in lista)`.*

In [120]:
def todos_sao_sensor(lista):
    # TODO: devolva True se TODOS os itens forem RoboComSensor
    return all(isinstance(r, RoboComSensor) for r in lista)
    


print(todos_sao_sensor(robos))
print(todos_sao_sensor([RoboComSensor("X"), RoboComSensor("Y")]))

False
True


## Composição — `self.outro = OutraClasse(...)`

Composição é mais simples que herança, na sintaxe: um atributo que é, ele mesmo,
um objeto de outra classe. Não existe `class Carro(Motor)` em lugar nenhum — um
`Carro` não **é um** `Motor`, ele **tem um** `Motor`. No robô, `Sensor` vira uma
peça que `Robo` tem, e `sensor_frente()` **delega** pro `self.sensor.ler(self)`.

In [121]:
class Motor:
    def __init__(self, potencia):
        self.potencia = potencia
        self.ligado = False

    def ligar(self):
        self.ligado = True


class Carro:
    def __init__(self, marca, potencia_motor):
        self.marca = marca
        self.motor = Motor(potencia_motor)      # Carro TEM um Motor

    def ligar_carro(self):
        self.motor.ligar()


hrv = Carro("Honda", 150)
hrv.ligar_carro()
print(hrv.motor.ligado, hrv.motor.potencia)

True 150


In [122]:
from enum import Enum


class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


class Sensor:
    def __init__(self, alcance=1):
        self.alcance = alcance

    def ler(self, robo):
        dx, dy = robo.direcao.value
        for passo in range(1, self.alcance + 1):
            nx, ny = robo.x + dx * passo, robo.y + dy * passo
            if (nx, ny) in robo.obstaculos:
                return False
        return True


class Robo:
    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None, alcance_sensor=1):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}
        self.sensor = Sensor(alcance_sensor)     # Robo TEM um Sensor

    def sensor_frente(self):
        return self.sensor.ler(self)             # delega pro objeto composto


obstaculos = {(3, 0): True}
r1 = Robo("Wall-E", obstaculos=obstaculos, alcance_sensor=1)
r2 = Robo("R2D2", obstaculos=obstaculos, alcance_sensor=3)
print(r1.sensor_frente())
print(r2.sensor_frente())

True
False


### Sua vez

Complete `RoboComFarol.piscar()`: chame `self.farol.alternar()` — o `Farol` já
está composto em `__init__`, só falta delegar a chamada.

*Dica: uma linha — `self.farol.alternar()`.*

In [123]:
class Farol:
    def __init__(self, ligado=False):
        self.ligado = ligado

    def alternar(self):
        self.ligado = not self.ligado


class RoboComFarol(Robo):
    def __init__(self, nome, **kwargs):
        super().__init__(nome, **kwargs)
        self.farol = Farol()

    def piscar(self):
        # TODO: chame self.farol.alternar()
        self.farol.alternar()
        pass


rf = RoboComFarol("Bumblebee")
rf.piscar()
print(rf.farol.ligado)

True


## Composição também troca em tempo de execução

Herança fixa o tipo do objeto para sempre, desde `class Sub(Base):`. Composição é
só um atributo — pode ser **trocado** a qualquer momento, sem criar objeto novo.

In [124]:
print(r1.sensor_frente())

r1.sensor = Sensor(alcance=5)     # troca a peça, mesmo objeto
print(r1.sensor_frente())

True
False


In [125]:
class Radio:
    def __init__(self, alcance=5):
        self.alcance = alcance

    def transmitir(self, mensagem):
        return f"[alcance {self.alcance}] {mensagem}"


radio = Radio(10)
print(radio.transmitir("SOS"))

[alcance 10] SOS


### Sua vez

Complete `RoboComRadio.trocar_radio(novo_alcance)`: substitua `self.radio` por
um `Radio` **novo**, com o alcance passado.

*Dica: `self.radio = Radio(novo_alcance)`, mesmo padrão da troca de `sensor` acima.*

In [ ]:
class RoboComRadio(Robo):
    def __init__(self, nome, alcance_radio=5, **kwargs):
        super().__init__(nome, **kwargs)
        self.radio = Radio(alcance_radio)

    def trocar_radio(self, novo_alcance):
        # TODO: substitua self.radio por um Radio novo, com alcance novo_alcance
        self.radio = Radio(novo_alcance)
        pass


rr = RoboComRadio("Bender", alcance_radio=5, obstaculos=obstaculos)
print(rr.radio.alcance)
rr.trocar_radio(20)
print(rr.radio.alcance)

5
5


## Para aprofundar

- Herança (visão geral) — Tutorial oficial: https://docs.python.org/3/tutorial/classes.html#inheritance
- `super()` — documentação oficial: https://docs.python.org/3/library/functions.html#super
- `isinstance()` — documentação oficial: https://docs.python.org/3/library/functions.html#isinstance
- Composição vs. herança — Real Python: https://realpython.com/inheritance-composition-python/
- Ordem de resolução de métodos (MRO), para quem quiser ir além — documentação oficial: https://docs.python.org/3/library/stdtypes.html#class.__mro__